# FastAPI and Docker for Machine Learning & Data Science Learners

## 1. Introduction

If you are learning **Machine Learning (ML)** or **Data Science (DS)**, you will eventually face a common problem:

> "I trained a model in a notebook. How do I make that model available to another application, another person, or a production system?"

Two technologies that help solve this problem are:

- **FastAPI** — used to expose Python functionality, such as an ML model, through an HTTP API.
- **Docker** — used to package an application and its dependencies into a portable container.

A useful way to think about them is:

```text
Data → Model Training → Saved Model → FastAPI → Docker → Deployment
```

FastAPI and Docker are **not replacements for Python, pandas, NumPy, scikit-learn, PyTorch, or TensorFlow**. They sit around your ML code and help turn a model into a usable application.

---

# 2. The Machine Learning Learner's Journey

A typical beginner ML project looks like this:

```text
1. Collect data
       ↓
2. Clean and preprocess data
       ↓
3. Explore data
       ↓
4. Train model
       ↓
5. Evaluate model
       ↓
6. Save model
       ↓
7. Make predictions
```

In a notebook, you might do:

```python
prediction = model.predict([[25, 50000, 3]])
```

That works for you.

But suppose you want a website, mobile application, another Python program, or another developer to use the model.

They should not have to open your notebook and execute Python code manually.

Instead, you can create an API:

```text
Client
  |
  | POST /predict
  | {"age": 25, "income": 50000, "experience": 3}
  ↓
FastAPI
  |
  ↓
ML Model
  |
  ↓
Prediction
  |
  ↓
JSON Response
```

Then Docker can package the entire application:

```text
Docker Container
├── Python
├── FastAPI
├── ML libraries
├── Your application code
├── Saved ML model
└── Configuration
```

---

# 3. What is FastAPI?

**FastAPI** is a Python web framework for building APIs.

An API provides a way for one program to communicate with another.

For example:

```text
Client → API → Model → Prediction
```

A FastAPI application can expose an endpoint such as:

```text
POST /predict
```

The client sends data:

```json
{
  "age": 25,
  "income": 50000,
  "experience": 3
}
```

The API sends back:

```json
{
  "prediction": 72000
}
```

The important point for an ML learner is:

> FastAPI allows your Python ML code to become accessible through a standard HTTP interface.

---

# 4. Why Should an ML/Data Science Learner Learn FastAPI?

You do **not** need FastAPI to train a machine learning model.

For example, you can train a model using:

```text
pandas
NumPy
scikit-learn
PyTorch
TensorFlow
XGBoost
```

However, FastAPI becomes useful when you want to **serve the model**.

### Without FastAPI

```text
Notebook
   ↓
Python code
   ↓
Prediction
```

Only someone running your Python environment can easily use it.

### With FastAPI

```text
Website
Mobile App
Python Client
Another Service
     |
     ↓
  FastAPI
     |
     ↓
 ML Model
```

This separates the model from the application that consumes it.

---

# 5. Where Does FastAPI Belong in an ML Project?

FastAPI usually belongs in the **model serving / inference layer**.

A simplified ML lifecycle is:

```text
                    MACHINE LEARNING PROJECT

Data
 ↓
EDA
 ↓
Feature Engineering
 ↓
Training
 ↓
Evaluation
 ↓
Model Saving
 ↓
────────────────────────────
      Model Serving
────────────────────────────
 ↓
FastAPI
 ↓
Docker
 ↓
Cloud / Server
 ↓
Application
```

The important distinction is:

### Training

Training usually happens separately.

```python
model.fit(X_train, y_train)
```

### Inference

Inference happens when you use the trained model:

```python
prediction = model.predict(X)
```

FastAPI is generally used around the **inference** part.

---

# 6. Training vs Inference

Understanding this distinction is extremely important.

## Training

Training can be computationally expensive.

Example:

```text
10 million rows
       ↓
Feature engineering
       ↓
Model training
       ↓
Several hours
       ↓
model.pkl
```

## Inference

Inference is when a new user asks for a prediction.

```text
User input
    ↓
Preprocessing
    ↓
Saved model
    ↓
Prediction
```

For example:

```text
Age = 30
Income = 60000
Experience = 5
       ↓
Prediction = 85000
```

FastAPI is primarily useful for making this **inference process available to other systems**.

---

# 7. A Simple FastAPI Example

First install FastAPI and Uvicorn:

```bash
pip install fastapi uvicorn
```

Create:

```text
main.py
```

Example:

```python
from fastapi import FastAPI

app = FastAPI()


@app.get("/")
def home():
    return {"message": "ML API is running"}
```

Run it:

```bash
uvicorn main:app --reload
```

You can then access:

```text
http://127.0.0.1:8000
```

FastAPI also automatically provides interactive API documentation.

Usually:

```text
http://127.0.0.1:8000/docs
```

This is one of the reasons FastAPI is convenient for beginners.

---

# 8. Turning an ML Model into an API

Suppose you trained a simple regression model.

Your training code might look like:

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)
```

After training, save it:

```python
import joblib

joblib.dump(model, "model.pkl")
```

Now you have:

```text
model.pkl
```

This file contains your trained model.

Your API can load it:

```python
import joblib
from fastapi import FastAPI

app = FastAPI()

model = joblib.load("model.pkl")
```

Then create a prediction endpoint.

```python
from pydantic import BaseModel


class PredictionInput(BaseModel):
    age: float
    income: float
    experience: float


@app.post("/predict")
def predict(data: PredictionInput):

    features = [[
        data.age,
        data.income,
        data.experience
    ]]

    prediction = model.predict(features)

    return {
        "prediction": float(prediction[0])
    }
```

Now the architecture is:

```text
             HTTP Request
                  ↓
       ┌────────────────────┐
       │      FastAPI       │
       │                    │
       │  /predict          │
       └─────────┬──────────┘
                 ↓
             Input data
                 ↓
             ML Model
                 ↓
             Prediction
                 ↓
              JSON
```

---

# 9. Why Use Pydantic?

In the previous example:

```python
class PredictionInput(BaseModel):
    age: float
    income: float
    experience: float
```

Pydantic helps define and validate the expected input.

For example:

```json
{
  "age": 25,
  "income": 50000,
  "experience": 3
}
```

The API knows what fields it expects.

This is particularly useful for ML APIs because bad input can cause prediction errors.

For example:

```json
{
  "age": "hello",
  "income": 50000
}
```

The API can detect that the input does not match the expected structure.

---

# 10. What is Docker?

Docker is a technology for packaging an application and its dependencies into a **container**.

Imagine your ML project requires:

```text
Python 3.12
FastAPI
scikit-learn
pandas
NumPy
joblib
```

Your computer has all of these installed.

You send the project to another computer.

That computer might have:

```text
Python 3.10
different package versions
missing dependencies
different operating system configuration
```

Your application might fail.

Docker helps solve this problem by packaging the environment needed to run the application.

---

# 11. Docker in Simple Terms

Without Docker:

```text
Your Code
   +
Your Python
   +
Your Packages
   +
Your Environment
```

With Docker:

```text
┌─────────────────────────────┐
│       Docker Container      │
│                             │
│  Python                     │
│  FastAPI                    │
│  ML libraries               │
│  Application code           │
│  Saved model                │
│                             │
└─────────────────────────────┘
```

The container can then run on a machine that supports Docker.

---

# 12. Why Should an ML/Data Science Learner Learn Docker?

Docker becomes valuable when your project moves beyond experimentation.

As a beginner, you might have:

```text
Jupyter Notebook
```

Later:

```text
Python script
```

Then:

```text
FastAPI application
```

Then:

```text
Docker container
```

Then:

```text
Cloud deployment
```

Docker is therefore useful when you start thinking about **reproducibility and deployment**.

It also helps with a very common ML problem:

> "It works on my machine."

Docker aims to make the runtime environment explicit and reproducible.

---

# 13. FastAPI vs Docker

These technologies solve different problems.

| Technology | Main purpose |
|---|---|
| FastAPI | Build and expose an API |
| Docker | Package and run the application consistently |
| scikit-learn | Build ML models |
| pandas | Data manipulation |
| NumPy | Numerical computation |
| PostgreSQL | Store data |
| Git | Version control |
| Cloud platform | Run/deploy applications |

Think of them as different layers.

```text
             User/Application
                    ↓
                 FastAPI
                    ↓
              Python ML Code
                    ↓
                ML Model
                    ↓
              Docker Container
                    ↓
               Cloud/Server
```

---

# 14. How FastAPI and Docker Work Together

Suppose your ML project has:

```text
ml-api/
│
├── main.py
├── model.pkl
├── requirements.txt
└── Dockerfile
```

FastAPI handles the API.

Docker packages the application.

```text
             Client
               |
               | HTTP
               ↓
        ┌───────────────┐
        │ FastAPI       │
        │ Container     │
        │               │
        │ main.py       │
        │ model.pkl     │
        │ dependencies  │
        └───────┬───────┘
                ↓
            Prediction
```

---

# 15. requirements.txt

A Python project should normally specify its dependencies.

Example:

```text
fastapi
uvicorn
scikit-learn
joblib
numpy
pandas
```

A more reproducible project should pin versions when appropriate:

```text
fastapi==0.x.x
uvicorn==0.x.x
scikit-learn==1.x.x
joblib==1.x.x
numpy==2.x.x
pandas==2.x.x
```

The exact versions should match the versions with which your model and application were developed and tested.

---

# 16. Creating a Dockerfile

A Dockerfile describes how to build your container image.

Example:

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

Let's understand it line by line.

### Base image

```dockerfile
FROM python:3.12-slim
```

Start with a lightweight Python environment.

### Working directory

```dockerfile
WORKDIR /app
```

The application will live inside `/app`.

### Copy dependencies

```dockerfile
COPY requirements.txt .
```

Copy the requirements file into the image.

### Install dependencies

```dockerfile
RUN pip install --no-cache-dir -r requirements.txt
```

Install the required Python packages.

### Copy application

```dockerfile
COPY . .
```

Copy the project into the container.

### Expose the API port

```dockerfile
EXPOSE 8000
```

Document the port the application uses.

### Start FastAPI

```dockerfile
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

Start the API server.

---

# 17. Building the Docker Image

From the project directory:

```bash
docker build -t ml-api .
```

This creates a Docker image called:

```text
ml-api
```

Run it:

```bash
docker run -p 8000:8000 ml-api
```

The mapping:

```text
-p 8000:8000
```

means:

```text
Your computer port 8000
          ↓
Container port 8000
```

Now your API can be accessed from your computer.

---

# 18. A Complete Beginner ML API Project

A simple structure could be:

```text
ml-api/
│
├── app/
│   ├── __init__.py
│   ├── main.py
│   ├── schemas.py
│   └── model.py
│
├── models/
│   └── model.pkl
│
├── requirements.txt
├── Dockerfile
├── .dockerignore
└── README.md
```

A more realistic project may eventually become:

```text
ml-project/
│
├── data/
│
├── notebooks/
│
├── src/
│   ├── preprocessing/
│   ├── training/
│   └── evaluation/
│
├── models/
│
├── api/
│   ├── main.py
│   ├── schemas.py
│   └── services.py
│
├── tests/
│
├── Dockerfile
├── requirements.txt
└── README.md
```

The exact structure is flexible. The important concept is to separate:

```text
Data science experimentation
        ↓
ML training
        ↓
Model artifact
        ↓
API/inference
        ↓
Containerization
        ↓
Deployment
```

---

# 19. Where Should Training Code Live?

Do not automatically put everything into FastAPI.

For example, avoid creating an API endpoint that performs expensive model training every time someone sends a request.

Usually:

```text
Training Pipeline
       ↓
   model.pkl
       ↓
Inference API
       ↓
   Prediction
```

Training and serving are different responsibilities.

A training script might be:

```text
src/train.py
```

and the API might be:

```text
api/main.py
```

This separation makes the project easier to understand and maintain.

---

# 20. Where Should Data Preprocessing Live?

This depends on the project.

Suppose your training pipeline does:

```text
Raw data
   ↓
Missing value handling
   ↓
Encoding
   ↓
Scaling
   ↓
Model
```

The same transformations generally need to happen during inference.

A strong approach is to package preprocessing with the model when possible.

For example, with scikit-learn:

```python
from sklearn.pipeline import Pipeline
```

You can create:

```text
Preprocessing
      +
Model
      ↓
Pipeline
```

Then save the complete pipeline.

This helps avoid the common mistake of having different preprocessing during training and prediction.

---

# 21. Example: scikit-learn Pipeline

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

pipeline.fit(X_train, y_train)
```

Save it:

```python
import joblib

joblib.dump(pipeline, "model.pkl")
```

Then your API loads:

```python
pipeline = joblib.load("model.pkl")
```

The API can pass raw features into the pipeline.

This can make inference safer and simpler.

---

# 22. FastAPI Is Not the Model

This distinction is important.

FastAPI:

```text
Receives request
      ↓
Validates input
      ↓
Calls model
      ↓
Returns response
```

The ML model:

```text
Input features
      ↓
Prediction algorithm
      ↓
Prediction
```

Therefore:

```text
FastAPI ≠ Machine Learning

FastAPI = interface around Machine Learning
```

---

# 23. Docker Is Not Deployment

Another important distinction:

```text
Docker ≠ Cloud
```

Docker packages your application.

A cloud platform provides infrastructure where your container can run.

For example:

```text
Your ML application
        ↓
Docker image
        ↓
Container
        ↓
Cloud infrastructure
        ↓
Internet
```

You can use Docker locally without deploying anything to the cloud.

---

# 24. When Should an ML Learner Learn FastAPI?

A reasonable progression is:

### Stage 1 — Python

Learn:

```text
Python basics
Functions
Classes
Modules
Virtual environments
Exceptions
File handling
```

### Stage 2 — Data Science

Learn:

```text
NumPy
pandas
Matplotlib
statistics
SQL
EDA
```

### Stage 3 — Machine Learning

Learn:

```text
scikit-learn
Regression
Classification
Clustering
Feature engineering
Model evaluation
Cross-validation
```

### Stage 4 — Model Engineering

Learn:

```text
Model serialization
Inference
Pipelines
Project structure
Testing
```

### Stage 5 — API Development

Learn:

```text
HTTP
REST
JSON
FastAPI
Pydantic
API validation
```

### Stage 6 — Containerization

Learn:

```text
Docker
Dockerfile
Images
Containers
Volumes
Ports
Docker Compose
```

### Stage 7 — Deployment

Learn:

```text
Cloud basics
CI/CD
Monitoring
Logging
Model/version management
```

You do not need to master all of DevOps before building your first FastAPI ML project.

---

# 25. When Should You NOT Use FastAPI?

FastAPI is not necessary for every ML project.

If you are:

- Exploring a dataset
- Learning statistics
- Building a notebook
- Comparing algorithms
- Practicing feature engineering
- Doing an offline analysis

then you probably do not need FastAPI.

For example:

```text
EDA project
↓
Jupyter Notebook
↓
Report
```

is perfectly fine.

Use FastAPI when you have a reason to expose functionality through an API.

---

# 26. When Should You NOT Use Docker?

You also do not need Docker for every small experiment.

If you are:

```text
Learning pandas
Testing a regression algorithm
Doing a small Kaggle experiment
Exploring a dataset
```

Docker may add unnecessary complexity.

But when your project becomes:

```text
Reusable
Shareable
Deployable
Collaborative
Production-oriented
```

Docker becomes increasingly useful.

---

# 27. A Practical Example

Imagine you build a house-price prediction model.

### Step 1 — Data

```text
house_data.csv
```

contains:

```text
area
bedrooms
location
age
price
```

### Step 2 — Train

You train:

```text
RandomForestRegressor
```

### Step 3 — Save

```text
house_price_model.pkl
```

### Step 4 — API

Create:

```text
POST /predict
```

Input:

```json
{
  "area": 1200,
  "bedrooms": 3,
  "age": 10
}
```

Output:

```json
{
  "predicted_price": 185000
}
```

### Step 5 — Docker

Package:

```text
FastAPI
Python
scikit-learn
model
dependencies
```

into a container.

### Step 6 — Deployment

Run the container on a server/cloud environment.

The final architecture:

```text
                  User
                   |
                   ↓
              Web/Mobile App
                   |
                   ↓
                HTTP API
                   |
                   ↓
             ┌────────────┐
             │  FastAPI   │
             └─────┬──────┘
                   |
                   ↓
          Preprocessing Pipeline
                   |
                   ↓
             ML Model
                   |
                   ↓
              Prediction
                   |
                   ↓
                 JSON
```

And all of this can run inside:

```text
┌─────────────────────────────┐
│       Docker Container      │
│                             │
│ FastAPI                     │
│ Python                      │
│ Dependencies                │
│ Preprocessing               │
│ ML Model                    │
└─────────────────────────────┘
```

---

# 28. Important Concepts to Learn Alongside FastAPI

If you are an ML/Data Science learner, do not learn FastAPI in isolation.

Learn these concepts:

## HTTP

Understand:

```text
GET
POST
PUT
DELETE
```

For ML APIs, `POST` is especially common for sending prediction input.

---

## JSON

Learn how data is represented:

```json
{
  "age": 30,
  "income": 60000
}
```

---

## Status Codes

Know basic codes:

```text
200 → Success
201 → Created
400 → Bad request
404 → Not found
422 → Validation error
500 → Server error
```

---

## REST APIs

Understand the idea of resources and endpoints.

For example:

```text
GET  /health
POST /predict
GET  /model-info
```

---

# 29. Important Docker Concepts to Learn

You do not need every Docker feature initially.

Start with:

```text
Image
Container
Dockerfile
Port mapping
Volumes
Environment variables
Docker Hub
```

Then learn:

```text
Docker Compose
Networks
Multi-stage builds
Container health checks
Security
```

later.

---

# 30. Environment Variables

Never hard-code sensitive configuration into your code.

For example, instead of:

```python
API_KEY = "secret-key"
```

use environment variables.

Conceptually:

```text
Environment
    ↓
API_KEY
    ↓
Application
```

Docker can pass environment variables to containers.

This becomes particularly important when your ML application connects to:

```text
Databases
Cloud services
Object storage
External APIs
Authentication systems
```

---

# 31. Testing Your ML API

Once you have an API, test it.

You should test:

```text
Valid input
Invalid input
Missing fields
Wrong data types
Model errors
Unexpected values
```

For example:

```text
POST /predict
```

with:

```json
{
  "age": 30,
  "income": 60000,
  "experience": 5
}
```

Then test invalid input:

```json
{
  "age": "abc"
}
```

Testing is especially important for ML systems because the model may behave differently from ordinary business logic.

---

# 32. Model Versioning

Imagine you deploy:

```text
model_v1.pkl
```

Later you train a better model:

```text
model_v2.pkl
```

You should know which model produced which prediction.

A basic strategy:

```text
models/
├── model_v1.pkl
├── model_v2.pkl
└── current_model.pkl
```

More mature ML systems use dedicated model registries and experiment tracking systems.

You do not need those immediately, but the concept of **model versioning** is important to learn.

---

# 33. Common Beginner Mistakes

## Mistake 1: Training inside every API request

Avoid:

```text
POST /predict
    ↓
Train model
    ↓
Predict
```

Usually you want:

```text
Training
    ↓
Save model
    ↓
API loads model
    ↓
Predict
```

---

## Mistake 2: Forgetting preprocessing

If the model was trained using:

```text
scaling
encoding
feature selection
```

you need to reproduce the same transformations during inference.

Using a pipeline can help.

---

## Mistake 3: Treating Docker as magic

Docker does not automatically fix bad code.

It packages your application and environment.

You still need:

```text
Good code
Correct dependencies
Tests
Configuration
Security
Monitoring
```

---

## Mistake 4: Putting everything into one file

Avoid making:

```text
main.py
```

contain:

```text
data loading
training
preprocessing
model
API
database
logging
```

As your project grows, separate responsibilities.

---

## Mistake 5: Learning Docker before understanding Python

If you are still struggling with:

```python
functions
imports
classes
virtual environments
packages
```

focus on Python first.

Docker will make much more sense afterward.

---

# 34. Recommended Learning Project

A very good beginner project is:

## "House Price Prediction API"

Build it in stages.

### Stage 1

Train a model in a notebook.

```text
notebooks/
└── house_price.ipynb
```

### Stage 2

Move training into Python.

```text
src/
└── train.py
```

### Stage 3

Save the model.

```text
models/
└── house_price_model.pkl
```

### Stage 4

Create FastAPI.

```text
api/
└── main.py
```

### Stage 5

Create:

```text
POST /predict
```

### Stage 6

Validate input using Pydantic.

### Stage 7

Test the API.

### Stage 8

Create a Dockerfile.

### Stage 9

Build the Docker image.

### Stage 10

Run the container locally.

### Stage 11

Optionally deploy it to a cloud service.

This single project teaches a surprisingly large number of practical concepts.

---

# 35. Suggested Final Project Architecture

After completing the project, aim for something similar to:

```text
house-price-ml-api/
│
├── data/
│   └── raw/
│
├── notebooks/
│   └── exploration.ipynb
│
├── src/
│   ├── train.py
│   └── preprocessing.py
│
├── models/
│   └── house_price_model.pkl
│
├── app/
│   ├── __init__.py
│   ├── main.py
│   ├── schemas.py
│   └── predictor.py
│
├── tests/
│   └── test_api.py
│
├── requirements.txt
├── Dockerfile
├── .dockerignore
└── README.md
```

This is much closer to how you should think about an ML project than a single notebook containing everything.

---

# 36. The Bigger Picture

As an ML/Data Science learner, your goal should not be:

> "I need to learn FastAPI because everyone says it is important."

Instead, understand the problem it solves.

The progression is:

```text
                 DATA SCIENCE
                      |
                      ↓
              Explore the data
                      |
                      ↓
              Train ML model
                      |
                      ↓
             Evaluate the model
                      |
                      ↓
              Save model artifact
                      |
                      ↓
                 INFERENCE
                      |
                      ↓
                 FastAPI
                      |
                      ↓
               Docker container
                      |
                      ↓
                  Deployment
                      |
                      ↓
                  End users
```

FastAPI and Docker become useful when your ML work needs to move from **experimentation to usable software**.

---

# 37. What You Should Learn First

If you are currently a beginner, use this order:

```text
Python
  ↓
NumPy + pandas
  ↓
Statistics
  ↓
SQL
  ↓
Data Visualization
  ↓
Machine Learning
  ↓
scikit-learn
  ↓
Git/GitHub
  ↓
Virtual Environments
  ↓
Model Serialization
  ↓
HTTP + JSON
  ↓
FastAPI
  ↓
Docker
  ↓
Cloud Deployment
```

Do not rush into Docker and cloud deployment before you can comfortably build and explain a basic ML model.

---

# 38. FastAPI + Docker Cheat Sheet

## FastAPI

Install:

```bash
pip install fastapi uvicorn
```

Create app:

```python
from fastapi import FastAPI

app = FastAPI()
```

Run:

```bash
uvicorn main:app --reload
```

API documentation:

```text
/docs
```

Prediction endpoint:

```python
@app.post("/predict")
def predict(data):
    ...
```

---

## Docker

Build:

```bash
docker build -t ml-api .
```

Run:

```bash
docker run -p 8000:8000 ml-api
```

List containers:

```bash
docker ps
```

Stop a container:

```bash
docker stop <container_id>
```

List images:

```bash
docker images
```

---

# 39. Final Mental Model

Remember these three ideas:

### Machine Learning

> **Build the model.**

```text
Data → Training → Model
```

### FastAPI

> **Give the model an interface.**

```text
Request → API → Model → Response
```

### Docker

> **Package the application so it can run consistently.**

```text
Code + Dependencies + Model
              ↓
          Container
```

Together:

```text
                MACHINE LEARNING
                      │
                      ▼
                 Saved Model
                      │
                      ▼
                  FastAPI
                      │
                      ▼
                   Docker
                      │
                      ▼
                 Deployment
                      │
                      ▼
                 Real Users
```

That is the key relationship you should remember.

---

# 40. Conclusion

For a Data Science or Machine Learning learner, FastAPI and Docker should be viewed as **engineering tools around your ML work**, not as replacements for core ML knowledge.

You should first learn how to:

```text
understand data
    ↓
prepare data
    ↓
train models
    ↓
evaluate models
```

Then learn how to:

```text
save models
    ↓
serve predictions
    ↓
build APIs
    ↓
containerize applications
    ↓
deploy them
```

FastAPI answers:

> **"How can another application communicate with my ML model?"**

Docker answers:

> **"How can I package my application and its environment so it runs consistently elsewhere?"**

Once you understand these two questions, the role of FastAPI and Docker in an ML project becomes much clearer.

---

## Recommended First Hands-On Project

Build this:

```text
House Price Prediction API
```

with:

```text
Python
      +
pandas
      +
scikit-learn
      +
joblib
      +
FastAPI
      +
Pydantic
      +
Docker
```

Your final project should support:

```text
POST /predict
```

and accept:

```json
{
  "area": 1200,
  "bedrooms": 3,
  "age": 10
}
```

and return:

```json
{
  "predicted_price": 185000
}
```

That project will give you a practical bridge from **Data Science → Machine Learning → ML Engineering → Deployment**.
